# Modality Heterogeneity

## Problem
The standard FedAvg algorithm requires that every client has the exact same model architecture. If Client 1 (sensor-only) has an MLP and Client 2 (camera-only) has a CNN, you cannot average their weights.

To simulate this "real-world scenario," we need a more advanced approach. The most common solution is to have one, large, multi-modal global model, and clients only download and train the parts (or "branches") of the model for which they have data.


## The "Real-World" Scenario
We will create 8 clients (from 8 different subjects) with different data availabilities:

Clients 0, 1 (Full): Have all modalities (Sensor + Camera 1 + Camera 2).

Clients 2, 3 (Sensor Only): Only have sensor data. Their cameras are "off."

Clients 4, 5 (Sensor + Cam 1): Have sensor data and one camera, but the second camera is "broken."

Clients 6, 7 (Camera Only): Only have camera data. Their sensors are "off."

## Step 1: The Foundation (Imports and Setup)
Every Python script starts with importing the necessary libraries and setting up the environment. This is like laying the foundation for a house.

In [10]:
import random
import csv
from datetime import datetime
from torch.utils.data import Dataset
import pandas as pd
import os
import numpy as np
import seaborn as sns
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import sys

# -- Basic Setup --
# Set the device to use the GPU if available, otherwise use the CPU.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


### Define dataset loader

PyTorch uses a Dataset object to handle data loading. Since our model will take three different kinds of input (sensor data, image 1, and image 2), we need to create a special class that tells PyTorch how to retrieve one sample of each, along with its corresponding label.

This class will have three essential methods:

__init__: Initializes the dataset by storing our feature and label arrays.

__len__: Returns the total number of samples in the dataset.

__getitem__: Fetches a single data sample at a given index.

Here is the code for it. Add this to your script:

In [11]:
# Define dataset loader
class CustomDatasetRes(Dataset):
    def __init__(self, features1, features2, features3, labels):
        self.features1 = features1
        self.features2 = features2
        self.features3 = features3
        self.labels = labels

    def __len__(self):
        return len(self.features1)
    
    def __getitem__(self, index):
        return self.features1[index], self.features2[index], self.features3[index], self.labels[index]
    
# Define a simplified dataset loader for sensor data only
class SensorDataset(Dataset):
    def __init__(self, features, labels):
        self.features = features
        self.labels = labels

    def __len__(self):
        return len(self.features)
    
    def __getitem__(self, index):
        return self.features[index], self.labels[index]

### Helper Functions
Next, we'll add a few helper functions. These functions will perform common tasks that we'll need later, like displaying results, scaling data, and ensuring our experiments are reproducible.

1. display_result

This function takes the true labels (y_test) and the model's predicted labels (y_pred) and prints out standard performance metrics like accuracy, precision, recall, and F1-score.

In [12]:
def display_result(y_test, y_pred):
    print('Accuracy score : ', accuracy_score(y_test, y_pred))
    print('Precision score : ', precision_score(y_test, y_pred, average='weighted'))
    print('Recall score : ', recall_score(y_test, y_pred, average='weighted'))
    print('F1 score : ', f1_score(y_test, y_pred, average='weighted'))

2. scaled_data

This function uses Scikit-learn's StandardScaler to normalize the sensor (CSV) data. Scaling is crucial because it ensures that features with larger value ranges don't dominate the learning process. Notice there are two functions with the same name in the original code. In Python, the last definition of a function is the one that gets used. We will add both for completeness, but just know that the first one is effectively overwritten by the second.

In [13]:
def scale_data(X_train, X_test):
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    return X_train_scaled, X_test_scaled

def scaled_data(X_train):
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    return X_train_scaled

3. set_seed

This is a very important function for reproducibility. Machine learning involves a lot of randomness (e.g., initializing model weights, shuffling data). By setting a "seed," we ensure that the sequence of random numbers is the same every time we run the code, which means we'll get the exact same results.

In [14]:
def set_seed(seed=0):
    # Sets the environment variable for Python's hash seed
    os.environ['PYTHONHASHSEED'] = str(seed)
    # Sets the seed for NumPy's random number generator
    np.random.seed(seed)
    # Sets the seed for Python's built-in random module
    random.seed(seed)
    # Sets the seed for PyTorch's random number generator
    torch.manual_seed(seed)
    # If using a GPU, sets the seed for all CUDA devices
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)  # For multi-GPU setups
    # Ensures deterministic behavior in cuDNN (CUDA Deep Neural Network library)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

### loading and preprocessing the data.

The function loadClientsData is designed for a federated learning scenario. It reads data from separate files for each participant (or "client"), cleans it, aligns the different data types (sensor vs. image), and splits it into training and testing sets for each client.

Because this function is quite long, we'll build it in a few parts.

#### Part 1: Initializing and Processing Training Data
First, we'll define the function, list the subject IDs we want to load, and create empty dictionaries to store each client's data. Then, we'll start a loop to process each subject one by one. Inside the loop, we'll begin by loading and cleaning the training data.

This involves:

Reading the sensor data from a CSV file.

Removing rows with missing values and any duplicate rows.

Dropping columns that we don't need (like the 'Infrared' sensor readings).

Loading the corresponding image, label, and timestamp data from .npy files.

#### Part 2: Aligning and Preparing Training Data
After loading the raw data, we face a common problem: the datasets don't perfectly match. Because we dropped rows with missing values from the sensor (CSV) data, there are now timestamps in our image data that no longer have a corresponding entry in the sensor data.

We need to align them by removing the image samples that don't have a matching sensor reading.

After alignment, we'll prepare the data for the model:

Set the seed for reproducibility.

Separate features from labels.

One-hot encode the labels, converting them into a format suitable for the model's output layer (e.g., class 3 becomes [0, 0, 0, 1, 0, ...]).

Scale the numeric sensor data and the image pixel values.

Reshape the images to the format expected by the convolutional layers.

#### Part 3: Processing the Test Data and Finalizing the Function
The logic here is identical to what we just did for the training data:

Load the test sensor data (_test.csv) and test image data (_test.npy).

Clean the sensor data by removing missing values and unnecessary columns.

Align the test image data with the cleaned test sensor data.

Prepare the aligned test data (one-hot encode labels, scale features, reshape images).

Store all the processed training and test arrays into our dictionaries.

Increment the clint_index and repeat the process for the next subject.

After the loop finishes, the function returns all the dictionaries containing the data for every client.

### load_multimodal_clients_by_subject()
Here is the load_multimodal_clients_by_subject() function.

This function is designed to:

Load the main CSV, Camera 1, and Camera 2 data files.

Find the timestamps that are common across all three modalities.

Create one large, perfectly aligned master dataset.

Define clients based on a list of subjects (e.g., [1, 3, 4]).

Split the master dataset by subject to create each client's local training and validation data.

Return all the data splits needed for the simulation.

In [15]:
import pandas as pd
import numpy as np
import torch
from sklearn.preprocessing import StandardScaler
from torch.utils.data import Dataset
import os
import random

def load_heterogeneous_clients(csv_path, img_path, num_classes=12):
    """
    Loads data for all 17 subjects, creating 13 heterogeneous training clients
    and 1 global test set from the 4 test subjects.
    
    Each client = 1 subject.
    Each client is assigned a different modality subset to simulate heterogeneity.
    """
    print("--- Starting Heterogeneous Multi-Modal Data Loading ---")
    
    # --- 1. Load ALL data sources into memory ---
    print("Loading all data sources...")
    df = pd.read_csv(csv_path, header=[0, 1])
    # Clean column headers
    cleaned_columns = []
    last_val = ''
    for col_l1, col_l2 in df.columns:
        if 'Unnamed' in col_l1: col_l1 = last_val
        else: last_val = col_l1.strip(); col_l1 = last_val
        if col_l1 == col_l2.strip(): cleaned_columns.append(col_l1)
        else: cleaned_columns.append(f"{col_l1}_{col_l2.strip()}")
    df.columns = cleaned_columns
    
    img1 = np.load(f'{img_path}/image_1.npy')
    name1 = np.load(f'{img_path}/name_1.npy').flatten() # Fix: Ensure 1D
    label1 = np.load(f'{img_path}/label_1.npy').flatten() # Fix: Ensure 1D
    
    img2 = np.load(f'{img_path}/image_2.npy')
    name2 = np.load(f'{img_path}/name_2.npy').flatten() # Fix: Ensure 1D
    label2 = np.load(f'{img_path}/label_2.npy').flatten() # Fix: Ensure 1D
    print("All data sources loaded.")

    # --- 2. Create global, timestamp-indexed DataFrames for alignment ---
    df_sensor = df.drop(columns=[col for col in df.columns if 'Infrared' in col] + ['Trial', 'Tag'], errors='ignore')
    df_sensor.dropna(inplace=True)
    df_sensor.drop_duplicates(subset=['TimeStamps_Time'], inplace=True)
    df_sensor = df_sensor.set_index('TimeStamps_Time').sort_index()
    
    # Image 1 data
    # Create with timestamp as a COLUMN
    df_img1 = pd.DataFrame({'timestamp': name1, 'img1': list(img1), 'label1': label1})
    # Drop duplicates based ONLY on the hashable 'timestamp' column
    df_img1.drop_duplicates(subset=['timestamp'], inplace=True)
    # NOW set the index
    df_img1 = df_img1.set_index('timestamp').sort_index()

    # Image 2 data
    # Create with timestamp as a COLUMN
    df_img2 = pd.DataFrame({'timestamp': name2, 'img2': list(img2), 'label2': label2})
    # Drop duplicates based ONLY on the hashable 'timestamp' column
    df_img2.drop_duplicates(subset=['timestamp'], inplace=True)
    # NOW set the index
    df_img2 = df_img2.set_index('timestamp').sort_index()
    '''# Verify labels match across modalities
    common_labels = df_sensor['Activity'].loc[df_sensor.index.intersection(df_img1.index)].loc[df_img2.index]
    if not (common_labels.equals(df_img1.loc[common_labels.index, 'label1']) and 
            common_labels.equals(df_img2.loc[common_labels.index, 'label2'])):
        raise ValueError("Label mismatch between sensor and image modalities!")'''
    
    # --- 3. Find Common Timestamps and Create Master DataFrame ---
    common_index = df_sensor.index.intersection(df_img1.index).intersection(df_img2.index)
    print(f"Found {len(common_index)} common timestamps across all 3 modalities.")
    
    aligned_df = df_sensor.loc[common_index].copy()
    aligned_df['img1'] = df_img1.loc[common_index, 'img1']
    aligned_df['img2'] = df_img2.loc[common_index, 'img2']
    aligned_df['Activity'] = aligned_df['Activity'].astype(int)
    aligned_df['Activity'] = aligned_df['Activity'].replace(20, 0) # Map 'Special Fall' to 0
    
    # Define all 36 sensor columns
    sensor_cols = [
        'AnkleAccelerometer_x-axis (g)', 'AnkleAccelerometer_y-axis (g)', 'AnkleAccelerometer_z-axis (g)', 
        'AnkleAngularVelocity_x-axis (deg/s)', 'AnkleAngularVelocity_y-axis (deg/s)', 'AnkleAngularVelocity_z-axis (deg/s)', 
        'AnkleLuminosity_illuminance (lx)',
        'RightPocketAccelerometer_x-axis (g)', 'RightPocketAccelerometer_y-axis (g)', 'RightPocketAccelerometer_z-axis (g)', 
        'RightPocketAngularVelocity_x-axis (deg/s)', 'RightPocketAngularVelocity_y-axis (deg/s)', 'RightPocketAngularVelocity_z-axis (deg/s)', 
        'RightPocketLuminosity_illuminance (lx)',
        'BeltAccelerometer_x-axis (g)', 'BeltAccelerometer_y-axis (g)', 'BeltAccelerometer_z-axis (g)', 
        'BeltAngularVelocity_x-axis (deg/s)', 'BeltAngularVelocity_y-axis (deg/s)', 'BeltAngularVelocity_z-axis (deg/s)', 
        'BeltLuminosity_illuminance (lx)',
        'NeckAccelerometer_x-axis (g)', 'NeckAccelerometer_y-axis (g)', 'NeckAccelerometer_z-axis (g)', 
        'NeckAngularVelocity_x-axis (deg/s)', 'NeckAngularVelocity_y-axis (deg/s)', 'NeckAngularVelocity_z-axis (deg/s)', 
        'NeckLuminosity_illuminance (lx)',
        'WristAccelerometer_x-axis (g)', 'WristAccelerometer_y-axis (g)', 'WristAccelerometer_z-axis (g)', 
        'WristAngularVelocity_x-axis (deg/s)', 'WristAngularVelocity_y-axis (deg/s)', 'WristAngularVelocity_z-axis (deg/s)', 
        'WristLuminosity_illuminance (lx)',
        'BrainSensor'
    ]
    sensor_cols = [col for col in sensor_cols if col in aligned_df.columns]
    print(f"Using {len(sensor_cols)} sensor features.") # Should be 36
    
    # --- 4. Define Train/Test Subjects ---
    all_subjects = aligned_df['Subject'].unique()
    # Subjects 1-13 (excluding 5, 9 which are not in the dataset)
    train_subjects = sorted([s for s in all_subjects if s <= 13])
    # Subjects 14-17
    test_subjects = sorted([s for s in all_subjects if s >= 14])
    print(f"Training Clients (Subjects): {train_subjects}")
    print(f"Test Set (Subjects): {test_subjects}")
    
    # --- 5. Create Global Train/Test DataFrames ---
    train_df = aligned_df[aligned_df['Subject'].isin(train_subjects)]
    test_df = aligned_df[aligned_df['Subject'].isin(test_subjects)]

    # --- 6. Preprocess and Scale Global Data ---
    print("Scaling and preprocessing data...")
    scaler = StandardScaler()
    train_csv_scaled = scaler.fit_transform(train_df[sensor_cols])
    test_csv_scaled = scaler.transform(test_df[sensor_cols])
    
    train_img1_scaled = np.stack(train_df['img1'].values).reshape(-1, 32, 32, 1) / 255.0
    test_img1_scaled = np.stack(test_df['img1'].values).reshape(-1, 32, 32, 1) / 255.0
    
    train_img2_scaled = np.stack(train_df['img2'].values).reshape(-1, 32, 32, 1) / 255.0
    test_img2_scaled = np.stack(test_df['img2'].values).reshape(-1, 32, 32, 1) / 255.0

    set_seed()
    Y_train = torch.nn.functional.one_hot(torch.from_numpy(train_df['Activity'].values).long(), num_classes).float()
    Y_test = torch.nn.functional.one_hot(torch.from_numpy(test_df['Activity'].values).long(), num_classes).float()
    
    # --- 7. Define Modality Combinations ---
    # We have 7 valid combinations (excluding "no data")
    modality_options = [
        {'csv': True, 'img1': True, 'img2': True},   # Full
        {'csv': True, 'img1': False, 'img2': False}, # Sensor only
        {'csv': False, 'img1': True, 'img2': True},  # Camera only
        {'csv': True, 'img1': True, 'img2': False},  # Sensor + Cam1
        {'csv': True, 'img1': False, 'img2': True},  # Sensor + Cam2
        {'csv': False, 'img1': True, 'img2': False}, # Cam1 only
        {'csv': False, 'img1': False, 'img2': True}  # Cam2 only
    ]
    
    # --- 8. Create Client Splits & Assign Modalities ---
    print("Creating client splits and assigning heterogeneous modalities...")
    X_train_splits_csv, X_test_splits_csv = {}, {}
    X_train_splits_img1, X_test_splits_img1 = {}, {}
    X_train_splits_img2, X_test_splits_img2 = {}, {}
    Y_train_splits, Y_test_splits = {}, {}
    
    client_info = {}
    client_modality_configs = [] # This will be our list of configs

    client_id_counter = 0
    for sub in train_subjects:
        indices_train = (train_df['Subject'] == sub)
        if np.sum(indices_train) == 0: continue # Should not happen, but safe to check
            
        # Assign data
        X_train_splits_csv[client_id_counter] = train_csv_scaled[indices_train]
        X_train_splits_img1[client_id_counter] = train_img1_scaled[indices_train]
        X_train_splits_img2[client_id_counter] = train_img2_scaled[indices_train]
        Y_train_splits[client_id_counter] = Y_train[indices_train]
        
        # Assign this client a small, random 10% slice of the *global* test set
        # for its local validation.
        test_size = len(test_df)
        val_indices = np.random.choice(test_size, int(test_size * 0.1), replace=False)
        
        X_test_splits_csv[client_id_counter] = test_csv_scaled[val_indices]
        X_test_splits_img1[client_id_counter] = test_img1_scaled[val_indices]
        X_test_splits_img2[client_id_counter] = test_img2_scaled[val_indices]
        Y_test_splits[client_id_counter] = Y_test[val_indices]

        # Assign a modality by cycling through the 7 options
        config = modality_options[client_id_counter % len(modality_options)]
        client_modality_configs.append(config)
        client_info[client_id_counter] = f"Subject_{sub}"
        
        print(f"  - Client {client_id_counter} (Subject {sub}): Modalities={config}")
        
        client_id_counter += 1

    print(f"--- Successfully created {len(client_info)} heterogeneous clients. ---")

    # The Global Test Set uses ALL data from test subjects
    global_test_set = [test_csv_scaled, test_img1_scaled, test_img2_scaled, Y_test]
    
    return X_train_splits_csv, X_train_splits_img1, X_train_splits_img2, Y_train_splits, \
           X_test_splits_csv, X_test_splits_img1, X_test_splits_img2, Y_test_splits, \
           global_test_set, client_info, len(sensor_cols), client_modality_configs

In [16]:
# --- Hyperparameters ---
max_acc = 1
epoch = 50
epoch_size = 64
num_clients_to_select = 5 # Select 5 clients per round from the 13
local_epoch_per_round = 3
round_early_stop = 10
num_classes = 12 # 0-11
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Define paths
csv_path = '/home/syed/PhD/UP_Fall_Dataset/Sensor + Image/sensor.csv'
img_path = '/home/syed/PhD/UP_Fall_Dataset/Sensor + Image'

# Load data using the new heterogeneous function
# It now returns the configs list automatically
X_train_csv, X_train_img1, X_train_img2, Y_train, \
X_test_csv, X_test_img1, X_test_img2, Y_test, \
global_test_set, client_info, num_csv_features, \
client_modality_configs = load_heterogeneous_clients(
    csv_path=csv_path,
    img_path=img_path,
    num_classes=num_classes
)

--- Starting Heterogeneous Multi-Modal Data Loading ---
Loading all data sources...
All data sources loaded.
Found 258113 common timestamps across all 3 modalities.
Using 36 sensor features.
Training Clients (Subjects): [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(6), np.int64(7), np.int64(8), np.int64(10), np.int64(11), np.int64(12), np.int64(13)]
Test Set (Subjects): [np.int64(14), np.int64(15), np.int64(16), np.int64(17)]
Scaling and preprocessing data...
Creating client splits and assigning heterogeneous modalities...


/tmp/ipykernel_226133/149740253.py:160: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  Y_train_splits[client_id_counter] = Y_train[indices_train]


  - Client 0 (Subject 1): Modalities={'csv': True, 'img1': True, 'img2': True}
  - Client 1 (Subject 2): Modalities={'csv': True, 'img1': False, 'img2': False}
  - Client 2 (Subject 3): Modalities={'csv': False, 'img1': True, 'img2': True}
  - Client 3 (Subject 4): Modalities={'csv': True, 'img1': True, 'img2': False}
  - Client 4 (Subject 6): Modalities={'csv': True, 'img1': False, 'img2': True}
  - Client 5 (Subject 7): Modalities={'csv': False, 'img1': True, 'img2': False}
  - Client 6 (Subject 8): Modalities={'csv': False, 'img1': False, 'img2': True}
  - Client 7 (Subject 10): Modalities={'csv': True, 'img1': True, 'img2': True}
  - Client 8 (Subject 11): Modalities={'csv': True, 'img1': False, 'img2': False}
  - Client 9 (Subject 12): Modalities={'csv': False, 'img1': True, 'img2': True}
  - Client 10 (Subject 13): Modalities={'csv': True, 'img1': True, 'img2': False}
--- Successfully created 11 heterogeneous clients. ---


## Step 2: Client Selection

We're making great progress. We've handled all the data loading and preparation. Now, we'll add the functions that form the "intelligence" of our federated learning system: client selection.

Instead of blindly averaging updates from every client in every round, these methods evaluate each client's performance and contribution. This allows the server to select the most promising or reliable clients to participate in the global model update, potentially leading to faster convergence and a more robust final model.

We'll add a series of functions, each calculating a specific metric to judge the clients.

### Client Evaluation Metrics
Add all the following functions to your script. Each one calculates a different score based on a client's performance.

1. Relative Loss Reduction (RF_loss)

This measures how much a client's training loss has dropped from the beginning to the end of a local training round, relative to the client with the biggest drop. A higher score means the client is learning effectively.

In [17]:
def calculate_relative_loss_reduction_as_list(client_losses):
    """
    Calculates the relative loss reduction (RF_loss) for each client.
    """
    loss_reduction = {}
    for client_id, losses in client_losses.items():
        if len(losses) < 2:
            raise ValueError(f"Client {client_id} has less than 2 loss values, cannot calculate RF_loss.")
        loss_start = losses[0]
        loss_end = losses[-1]
        loss_reduction[client_id] = loss_start - loss_end

    max_loss_reduction = max(loss_reduction.values())
    if max_loss_reduction == 0:
        return [0.0] * len(loss_reduction)  # If no loss reduction, return 0.0 for all clients

    rf_losses_list = [
        reduction / max_loss_reduction for reduction in loss_reduction.values()
    ]
    return rf_losses_list

2. Relative Training Accuracy (RF_ACC_Train)

This measures a client's local training accuracy relative to the client with the highest accuracy. It's a straightforward measure of performance on local data.

In [18]:
def calculate_relative_train_accuracy(client_acc):
    """
    Calculates the relative training accuracy (RF_Acc_Train) for each client.
    """
    max_acc = max(client_acc.values())
    if max_acc == 0:
        return [0.0] * len(client_acc)  # If no accuracy, return 0.0 for all clients

    rf_accs_train_list = [
        acc / max_acc for acc in client_acc.values()
    ]
    return rf_accs_train_list

3. Global Validation Accuracy (RF_ACC_Global)

This is a more sophisticated metric. It rewards clients for high accuracy on a global test set but penalizes them if their global accuracy is much worse than their local training accuracy (which is a sign of overfitting).

In [19]:
def calculate_global_validation_accuracy(train_acc, global_acc):
    """
    Calculates the global validation accuracy (RF_Acc_Global) based on local training accuracies.
    """
    if set(train_acc.keys()) != set(global_acc.keys()):
        raise ValueError("Client IDs for train and global accuracy do not match.")

    max_global_acc = max(global_acc.values())
    if max_global_acc == 0:
        max_global_acc = 1  # Avoid division by zero

    global_train_diff = {
        client_id: train_acc[client_id] - global_acc[client_id]
        for client_id in train_acc
    }
    max_global_train_diff = max(global_train_diff.values())
    if max_global_train_diff == 0:
        max_global_train_diff = 1  # Avoid division by zero

    rf_acc_global_list = [
        (global_acc[client_id] / max_global_acc) - (global_train_diff[client_id] / max_global_train_diff)
        for client_id in train_acc
    ]
    return rf_acc_global_list

In [20]:
def calculate_relative_validation_accuracy(client_acc):
    """
    Calculates the relative validation accuracy (RF_ACC_Val) for each client.
    """
    # Ensure client_acc is a dictionary, not a list of lists
    if not isinstance(client_acc, dict):
        raise TypeError("Input must be a dictionary of client accuracies.")
        
    max_acc = max(client_acc.values())
    if max_acc == 0:
        return [0.0] * len(client_acc)

    return [acc / max_acc for acc in client_acc.values()]

4. Loss Outliers (P_loss)

This function flags clients that are potential negative contributors. If a client's final training loss is significantly higher than the average loss of all clients, it gets a high penalty score. Otherwise, its penalty is zero.

In [21]:
def calculate_loss_outliers(client_losses, lambda_loss=1.5):
    """
    Calculates the loss outlier penalty (P_loss) for each client.
    """
    final_losses = {client_id: losses[-1] for client_id, losses in client_losses.items()}
    loss_values = np.array(list(final_losses.values()))

    mean_loss = np.mean(loss_values)
    std_loss = np.std(loss_values)

    threshold = mean_loss + lambda_loss * std_loss

    max_loss = np.max(loss_values)

    if max_loss == 0:
        return [0.0] * len(loss_values)

    # Identify outliers
    loss_outliers = [
        final_loss / max_loss if final_loss > threshold else 0.0
        for final_loss in loss_values
    ]
    return loss_outliers

5. Performance Bias (P_bias)

This metric calculates the gap between a client's performance on its own validation data versus its performance on the global validation data. A large gap might indicate that the client's local data is not representative of the overall data distribution.

In [22]:
def calculate_performance_bias(val_acc, global_acc):
    """
    Calculates the performance bias penalty (P_bias).
    """
    if set(val_acc.keys()) != set(global_acc.keys()):
        raise ValueError("Client IDs for validation and global accuracy do not match.")

    performance_bias_list = []
    for client_id in val_acc:
        val = val_acc[client_id]
        global_val = global_acc[client_id]
        max_val = max(val, global_val)

        if max_val == 0:
            performance_bias = 0
        else:
            performance_bias = abs(val - global_val) / max_val
        performance_bias_list.append(performance_bias)

    return performance_bias_list

Excellent. Now that we have the functions to score each client, we need the final step: the algorithms that use these scores to select which clients will participate in a given round.

### Client Selection Algorithms
1. Pareto Optimization

This is a powerful technique used when you have multiple, often conflicting, objectives. Instead of combining all metrics into one score, it tries to find a set of clients that represent the best possible trade-offs.

A client is considered "Pareto optimal" if no other client is better than it across all metrics. The algorithm first finds this set of optimal clients.

If there are more optimal clients than needed, it selects a random subset.

If there are fewer, it fills the remaining spots by picking the clients with the best-combined performance score.

In [23]:
def pareto_optimization(
    rf_loss, rf_acc_train, rf_acc_val, rf_acc_global, p_loss, p_bias, client_num,
):
    """
    实现 Pareto 优化，筛选节点。

    参数：
    - rf_loss (list): 局部训练损失相对下降幅度。
    - rf_acc_train (list): 局部训练精度。
    - rf_acc_val (list): 局部验证精度。
    - rf_acc_global (list): 全局验证精度。
    - p_loss (list): 损失异常。
    - p_bias (list): 性能偏离。
    - client_num (int): 要选出的节点数。

    返回：
    - selected_clients (list): 选中的 client ID（按输入顺序从 0 开始）。
    """
    print("=== Pareto Optimization: Start ===")
    print("Input rf_loss:", [f"{x:.2f}" for x in rf_loss])
    print("Input rf_acc_train:", [f"{x:.2f}" for x in rf_acc_train])
    print("Input rf_acc_val:", [f"{x:.2f}" for x in rf_acc_val])
    print("Input rf_acc_global:", [f"{x:.2f}" for x in rf_acc_global])
    print("Input p_loss:", [f"{x:.2f}" for x in p_loss])
    print("Input p_bias:", [f"{x:.2f}" for x in p_bias])
    print(f"Number of clients to select: {client_num}")

    # Ensure all arrays are numpy arrays
    rf_loss = np.array(list(rf_loss))
    rf_acc_train = rf_acc_train.detach().cpu().numpy() if isinstance(rf_acc_train, torch.Tensor) else np.array(rf_acc_train)
    rf_acc_val = rf_acc_val.detach().cpu().numpy() if isinstance(rf_acc_val, torch.Tensor) else np.array(rf_acc_val)
    rf_acc_global = rf_acc_global.detach().cpu().numpy() if isinstance(rf_acc_global, torch.Tensor) else np.array(rf_acc_global)
    p_loss = p_loss.detach().cpu().numpy() if isinstance(p_loss, torch.Tensor) else np.array(p_loss)
    p_bias = p_bias.detach().cpu().numpy() if isinstance(p_bias, torch.Tensor) else np.array(p_bias)

    print("Converted all inputs to numpy arrays.")

    # Construct data matrix
    data = np.array([rf_loss, rf_acc_train, rf_acc_val, rf_acc_global, -p_loss, -p_bias]).T
    print(f"Constructed data matrix for Pareto: shape={data.shape}")

    # Pareto front selection
    def is_dominated(point, others):
        """判断 point 是否被 others 支配"""
        return any(np.all(other >= point) and np.any(other > point) for other in others)

    pareto_indices = [
        i for i, point in enumerate(data) if not is_dominated(point, np.delete(data, i, axis=0))
    ]
    pareto_clients = pareto_indices
    print(f"Pareto front client indices: {pareto_clients}")

    # If more Pareto clients than needed, randomly select
    if len(pareto_clients) > client_num:
        selected = random.sample(pareto_clients, client_num)
        #pareto_clients.sort()  # Sort by index ascending
        #selected = pareto_clients[:client_num]
        print(f"More Pareto clients than needed. Randomly selected: {selected}")
        print("=== Pareto Optimization: End ===")
        return [int(x) for x in selected]

    # If fewer Pareto clients, fill with best scores
    remaining_slots = client_num - len(pareto_clients)
    pareto_scores = [0.4 * rf_loss[i] + 0.6 * rf_acc_global[i] for i in range(len(rf_loss))]
    print("Pareto scores for all clients:", [f"{x:.2f}" for x in pareto_scores])
    sorted_indices = np.argsort(pareto_scores)[::-1]  # Descending order
    print("Sorted indices by Pareto score:", [int(x) for x in sorted_indices])

    selected_clients = set(pareto_clients)
    print(f"Initial selected clients (Pareto front): {[int(x) for x in selected_clients]}")
    for i in sorted_indices:
        if len(selected_clients) >= client_num:
            break
        if i not in selected_clients:
            selected_clients.add(int(i))
            print(f"Added client {int(i)} to fill remaining slots.")
            # If we have filled all slots, we can stop
            if len(selected_clients) >= client_num:
                break

    print(f"Final selected clients: {[int(x) for x in selected_clients]}")
    print("=== Pareto Optimization: End ===")
    return [int(x) for x in selected_clients]

2. Weighted Sum Method (5RF)

This is a more straightforward approach. It calculates a single comprehensive score for each client by taking a weighted sum of all the metrics. Clients with the highest final scores are selected. The weights (0.2, 0.1, 0.3, etc.) determine the importance of each metric.

In [24]:
def get_top_clients_with5RF(rf_loss, rf_acc_train, rf_acc_val, rf_acc_global, p_loss, p_bias, client_num):
    rf_loss = np.array(list(rf_loss))
    rf_acc_train = np.array(rf_acc_train)
    rf_acc_val = np.array(rf_acc_val)
    rf_acc_global = np.array(rf_acc_global)
    p_loss = np.array(p_loss)
    p_bias = np.array(p_bias)

    # Calculate a single weighted score for each client
    scores = (
            0.2 * rf_loss +
            0.1 * rf_acc_train +
            0.2 * rf_acc_val +
            0.3 * rf_acc_global -
            0.1 * p_loss -
            0.1 * p_bias
    )
    origin_scores = scores
    # Get the indices of the clients with the highest scores
    top_client_ids = np.argsort(scores)[::-1][:client_num]  # Sort descending and take the top N
    return top_client_ids.tolist(), origin_scores

## Step 3: The AI's Brain (The Model Definition)
We have the data pipeline and the client selection logic. Now it's time to build the brain of the operation: the neural network model itself.

The model, ModelCSVIMG, is a multi-modal neural network. This means it's designed to accept and process multiple types of data at once. It has three distinct input branches:

One for the numerical sensor (CSV) data.

One for the images from camera 1.

One for the images from camera 2.

The features extracted from each branch are then combined (fused) and passed to a final set of layers that perform the classification. The original code contains a few versions of the architecture; we will use the final, most complex one.

Add the complete model class to your script:

In [ ]:
# --- Branch 1: Sensor (CSV) Branch ---
class CsvBranch(nn.Module):
    def __init__(self, num_csv_features, out_features=600):
        super(CsvBranch, self).__init__()
        self.csv_fc_1 = nn.Linear(num_csv_features, 2000)
        self.csv_bn_1 = nn.BatchNorm1d(2000)
        self.csv_fc_2 = nn.Linear(2000, out_features)
        self.csv_bn_2 = nn.BatchNorm1d(out_features)
        self.csv_dropout = nn.Dropout(0.2)
    
    def forward(self, x_csv):
        x_csv = F.relu(self.csv_bn_1(self.csv_fc_1(x_csv)))
        x_csv = F.relu(self.csv_bn_2(self.csv_fc_2(x_csv)))
        return self.csv_dropout(x_csv)

# --- Branch 2 & 3: Image Branch (reusable) ---
class ImgBranch(nn.Module):
    def __init__(self, out_features=100):
        super(ImgBranch, self).__init__()
        self.conv_1 = nn.Conv2d(in_channels=1, out_channels=18, kernel_size=3, stride=1, padding=1)
        self.batch_norm = nn.BatchNorm2d(18)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.fc1 = nn.Linear(18 * 16 * 16, out_features) # 32x32 -> 16x16
        self.dropout = nn.Dropout(0.2)
    
    def forward(self, x_img):
        x_img = x_img.permute(0, 3, 1, 2) # (B, H, W, C) -> (B, C, H, W)
        x_img = F.relu(self.conv_1(x_img))
        x_img = self.batch_norm(x_img)
        x_img = self.pool(x_img)
        x_img = x_img.contiguous().view(x_img.size(0), -1) # Flatten
        x_img = F.relu(self.fc1(x_img))
        return self.dropout(x_img)

# --- Main Multi-Modal Model ---
class ModelCSVIMG(nn.Module):
    def __init__(self, num_csv_features, num_classes=12,
                 csv_out=600, img1_out=100, img2_out=100):
        super(ModelCSVIMG, self).__init__()
        
        self.csv_out = csv_out
        self.img1_out = img1_out
        self.img2_out = img2_out
        
        # --- Define Branches ---
        self.csv_branch = CsvBranch(num_csv_features, csv_out)
        self.img1_branch = ImgBranch(img1_out)
        self.img2_branch = ImgBranch(img2_out)
        
        # --- Fusion and Final Classification Layers ---
        self.fusion_input_size = csv_out + img1_out + img2_out # 800
        self.fc1 = nn.Linear(self.fusion_input_size, 1200)
        self.dr1 = nn.Dropout(0.2)
        self.fc2 = nn.Linear(self.fusion_input_size + 1200, num_classes) # Residual: 800 + 1200

    def forward(self, x_csv, x_img1, x_img2):
        # --- Process each branch ---
        # We process a branch if the input is not None
        # If input is None, we create a zero tensor as a placeholder
        
        batch_size = 0
        if x_csv is not None:
            batch_size = x_csv.size(0)
        elif x_img1 is not None:
            batch_size = x_img1.size(0)
        elif x_img2 is not None:
            batch_size = x_img2.size(0)
        
        if batch_size == 0:
             # This should not happen in a real batch
            return None 

        if x_csv is not None:
            csv_features = self.csv_branch(x_csv)
        else:
            csv_features = torch.zeros(batch_size, self.csv_out).to(device)

        if x_img1 is not None:
            img1_features = self.img1_branch(x_img1)
        else:
            img1_features = torch.zeros(batch_size, self.img1_out).to(device)
            
        if x_img2 is not None:
            img2_features = self.img2_branch(x_img2)
        else:
            img2_features = torch.zeros(batch_size, self.img2_out).to(device)

        # --- Fusion ---
        x = torch.cat((csv_features, img1_features, img2_features), dim=1)
        residual = x
        
        # --- Final layers ---
        x = F.relu(self.fc1(x))
        x = self.dr1(x)
        x = torch.cat((residual, x), dim=1)
        
        # Return logits (no softmax)
        x = self.fc2(x)
        return x

## Step 4: The Teacher (The Server Class)
Alright, we're on the home stretch. We have the data, the selection logic, and the model. Now we need to create the actors for our simulation: the Server and the Client. These two classes will control the entire federated learning process.

1. The Server Class

The Server is the central coordinator. Its job is to:

Hold the main global model.

Send the global model to the clients.

Receive updates from the selected clients.

Aggregate these updates to improve the global model.

Evaluate the global model's performance on a held-out test set.

Here is the code for the Server.

In [ ]:
# Server
class Server(object):
    def __init__(self, model, epoch_size, global_test_set, num_clients):
        self.global_model = model
        self.epoch_size = epoch_size
        self.num_clients_per_round = num_clients # Renamed for clarity
        
        self.serverTestDataSet = CustomDatasetRes(global_test_set[0], global_test_set[1], global_test_set[2], global_test_set[3])
        self.eval_loader = torch.utils.data.DataLoader(self.serverTestDataSet, batch_size=epoch_size)
        print(f"Server initialized with global test set of {len(self.serverTestDataSet)} samples.")

    def model_aggregate(self, selected_clients_configs, diff_client):
        """
        Performs smart, branch-wise aggregation.
        - selected_clients_configs: A list of configs, e.g., [client_0_config, client_1_config]
        - diff_client: The dictionary of weight diffs from all clients
        """
        
        # Keep track of how many clients trained each branch
        branch_counts = {'csv_branch': 0, 'img1_branch': 0, 'img2_branch': 0, 'fusion': 0}
        
        # Accumulate weights
        weight_accumulator = {name: torch.zeros_like(params) for name, params in self.global_model.state_dict().items()}
        
        for client_id, config in selected_clients_configs.items():
            # Check which branches this client trained
            trains_csv = config['csv']
            trains_img1 = config['img1']
            trains_img2 = config['img2']
            trains_fusion = (trains_csv + trains_img1 + trains_img2) >= 2
            
            # Add this client's diffs to the accumulator
            for name, diff_tensor in diff_client[client_id].items():
                if 'csv_branch' in name and trains_csv:
                    weight_accumulator[name].add_(diff_tensor)
                elif 'img1_branch' in name and trains_img1:
                    weight_accumulator[name].add_(diff_tensor)
                elif 'img2_branch' in name and trains_img2:
                    weight_accumulator[name].add_(diff_tensor)
                elif ('fc1' in name or 'dr1' in name or 'fc2' in name) and trains_fusion:
                    # This assumes fc1, dr1, fc2 are *only* in the fusion head
                    weight_accumulator[name].add_(diff_tensor)

            # Increment counts
            if trains_csv: branch_counts['csv_branch'] += 1
            if trains_img1: branch_counts['img1_branch'] += 1
            if trains_img2: branch_counts['img2_branch'] += 1
            if trains_fusion: branch_counts['fusion'] += 1

        print(f"Aggregation counts: {branch_counts}")

        # --- Apply the averaged updates to the global model ---
        for name, data in self.global_model.state_dict().items():
            update_tensor = None
            
            if 'csv_branch' in name and branch_counts['csv_branch'] > 0:
                update_tensor = weight_accumulator[name] / branch_counts['csv_branch']
            elif 'img1_branch' in name and branch_counts['img1_branch'] > 0:
                update_tensor = weight_accumulator[name] / branch_counts['img1_branch']
            elif 'img2_branch' in name and branch_counts['img2_branch'] > 0:
                update_tensor = weight_accumulator[name] / branch_counts['img2_branch']
            elif ('fc1' in name or 'dr1' in name or 'fc2' in name) and branch_counts['fusion'] > 0:
                update_tensor = weight_accumulator[name] / branch_counts['fusion']
            
            if update_tensor is not None:
                if data.dtype != update_tensor.dtype:
                    data.add_(update_tensor.to(data.dtype))
                else:
                    data.add_(update_tensor)

    def model_eval(self):
        self.global_model.eval()
        total_loss = 0.0
        correct = 0
        dataset_size = 0
        criterion = nn.CrossEntropyLoss(reduction='sum')
        with torch.no_grad():
            # In evaluation, we use ALL modalities
            for batch_id, (data1, data2, data3, target) in enumerate(self.eval_loader):
                dataset_size += data1.size()[0]
                
                data1 = data1.to(device).float()
                data2 = data2.to(device).float()
                data3 = data3.to(device).float()
                target = target.to(device).float()
                
                output = self.global_model(data1, data2, data3)
                
                total_loss += criterion(output, target).item()
                
                pred = output.detach().argmax(dim=1)
                target_indices = target.detach().argmax(dim=1)
                correct += pred.eq(target_indices.view_as(pred)).cpu().sum().item()

        acc = 100.0 * (float(correct) / float(dataset_size))
        loss = total_loss / float(dataset_size)
        return acc, loss

## The Client Class

2. The Client Class and Helper Functions

The Client represents an individual participant. Its job is to:

Receive the global model from the server.

Train this model on its own local data for a few epochs.

Calculate the change (the diff) between the original model and its newly trained model.

Send this diff back to the server.

The client's training process is handled by two helper functions: train_one_epoch and validate.

Add the Client class and its two helper functions to your script.

In [ ]:
# Client
class Client(object):
    def __init__(self, model, epoch_size, local_epoch_per_round, train_dataset, val_dataset, modality_config, id = -1):
        self.local_model = model # Use the global model instance
        self.epoch_size = epoch_size
        self.local_epoch_per_round = local_epoch_per_round
        self.client_id = id
        self.modality_config = modality_config # e.g., {'csv': True, 'img1': False, 'img2': True}
        
        # Use the CustomDatasetRes
        self.train_dataset = CustomDatasetRes(train_dataset[0], train_dataset[1], train_dataset[2], train_dataset[3])
        self.train_loader = torch.utils.data.DataLoader(self.train_dataset, batch_size=epoch_size, shuffle=True)
        
        self.eval_dataset = CustomDatasetRes(val_dataset[0], val_dataset[1], val_dataset[2], val_dataset[3])
        self.eval_loader = torch.utils.data.DataLoader(self.eval_dataset, batch_size=epoch_size, shuffle=False)
        
        print(f"Client {id}: {self.modality_config} - Train: {len(self.train_dataset)}, Val: {len(self.eval_dataset)}")

    def _set_trainable_layers(self):
        """Freezes/unfreezes model branches based on modality config."""
        for param in self.local_model.parameters():
            param.requires_grad = False # Freeze all by default
            
        # Unfreeze only the available branches
        if self.modality_config['csv']:
            for param in self.local_model.csv_branch.parameters():
                param.requires_grad = True
        
        if self.modality_config['img1']:
            for param in self.local_model.img1_branch.parameters():
                param.requires_grad = True
                
        if self.modality_config['img2']:
            for param in self.local_model.img2_branch.parameters():
                param.requires_grad = True

        # The fusion head is only trained if *all* modalities are present
        # This is a simplification. A more complex design could train it with partial inputs.
        # For now, we will train it if *at least two* modalities are present.
        if (self.modality_config['csv'] + self.modality_config['img1'] + self.modality_config['img2']) >= 2:
             for param in self.local_model.fc1.parameters():
                param.requires_grad = True
             for param in self.local_model.dr1.parameters():
                param.requires_grad = True
             for param in self.local_model.fc2.parameters():
                param.requires_grad = True

    def local_train(self, global_model):
        self.local_model.load_state_dict(global_model.state_dict())
        
        # --- KEY STEP: Freeze layers ---
        self._set_trainable_layers()

        criterion = nn.CrossEntropyLoss()
        # Only optimize parameters that require gradients
        optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, self.local_model.parameters()), lr=0.001)
        
        losses = []
        min_loss, max_loss = float('inf'), float('-inf')

        for epoch in range(self.local_epoch_per_round):
            train_loss, train_acc = train_one_epoch(self.local_model, self.train_loader, criterion, optimizer, self.modality_config)
            if train_loss > max_loss: max_loss = train_loss
            if train_loss < min_loss: min_loss = train_loss
            losses.append(train_loss)

        val_loss, val_acc = validate(self.local_model, self.eval_loader, criterion, self.modality_config)
        print(f"Client {self.client_id} - Train Acc: {train_acc:.2f}%, Val Acc: {val_acc:.2f}%")

        diff = dict()
        for name, data in self.local_model.state_dict().items():
            if data.requires_grad: # Only send back diffs for trained layers
                diff[name] = (data - global_model.state_dict()[name])
            
        return self.local_model, diff, val_acc, val_loss, min_loss, max_loss, losses, train_acc

# --- Updated Train/Validate Functions ---

def train_one_epoch(model, train_loader, criterion, optimizer, config):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for batch_id, (data1, data2, data3, target) in enumerate(train_loader):
        # Send data to device
        data1 = data1.to(device).float() if config['csv'] else None
        data2 = data2.to(device).float() if config['img1'] else None
        data3 = data3.to(device).float() if config['img2'] else None
        target = target.to(device).float()
        target_indices = target.argmax(dim=1)
        
        optimizer.zero_grad()
        output = model(data1, data2, data3)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * (data1.size(0) if data1 is not None else data2.size(0))
        predicted = output.argmax(dim=1)
        total += target.size(0)
        correct += predicted.eq(target_indices).sum().item()

    epoch_loss = running_loss / len(train_loader.dataset)
    epoch_acc = 100.0 * correct / total
    return epoch_loss, epoch_acc


def validate(model, val_loader, criterion, config):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for batch_id, (data1, data2, data3, target) in enumerate(val_loader):
            data1 = data1.to(device).float() if config['csv'] else None
            data2 = data2.to(device).float() if config['img1'] else None
            data3 = data3.to(device).float() if config['img2'] else None
            target = target.to(device).float()
            target_indices = target.argmax(dim=1)
            
            output = model(data1, data2, data3)
            loss = criterion(output, target)

            running_loss += loss.item() * (data1.size(0) if data1 is not None else data2.size(0))
            predicted = output.argmax(dim=1)
            total += target.size(0)
            correct += predicted.eq(target_indices).sum().item()

    epoch_loss = running_loss / len(val_loader.dataset)
    epoch_acc = 100.0 * correct / total
    return epoch_loss, epoch_acc

We are almost there! We've built all the major components. Before we assemble everything in the main training loop, we need to add the last few helper functions.

These functions are primarily used for a simpler, baseline client selection strategy (referred to as '4RF' in the code) that calculates a single score for each client and picks the best ones.

Add these final utility functions to your script.

1. Normalize

A standard function to scale any number to a range between 0 and 1, given a minimum and maximum value. This is useful for combining metrics that have different scales.

In [29]:
def normalize(value, min_value, max_value):
    # Avoid division by zero if min and max are the same
    if (max_value - min_value) == 0:
        return 0
    return (value - min_value) / (max_value - min_value)

2. Evaluate Model Score

This function calculates a simple, combined score for a client. It's a weighted average of their performance on their local training set and a global validation set.

In [30]:
def evaluate_model(acc, loss, min_loss, max_loss, oneclient_test_acc, oneclient_test_loss,alpha=0.8, beta=0.8):
    normalized_loss = normalize(loss, min_loss, max_loss)
    # Score based on local training performance
    train_score = alpha * acc + (1 - alpha) * (1 - normalized_loss) # Use (1 - loss) so higher is better

    # Score based on global validation performance
    val_score = beta * oneclient_test_acc + (1 - beta) * (1 - oneclient_test_loss)

    # Final combined score
    combined_score = (train_score + val_score) / 2
    return combined_score

3. Get Top Clients

A straightforward function that takes a dictionary of clients and their scores, then returns a list of the top num clients with the highest scores.

In [31]:
def get_top_clients(client_dict, num):
    # Sort the clients by their score (the dictionary value) in descending order
    sorted_clients = sorted(client_dict.items(), key=lambda item: item[1], reverse=True)
    # Extract just the IDs (the dictionary key) of the top clients
    top_clients = [client[0] for client in sorted_clients[:num]]
    return top_clients

4. Dynamic Threshold Selection

This is another, more advanced selection method included in the script. It selects clients whose scores are above a dynamic threshold (calculated from the mean and standard deviation of all scores). While not used in the final configuration, we include it for completeness.

In [32]:
def select_nodes_with_dynamic_threshold(node_scores, max_nodes, std_multiplier=1.0):
    """
    Selects nodes using a dynamic threshold based on score distribution.
    """
    if not node_scores:
        return []
        
    scores = np.array(list(node_scores.values()))
    
    # Calculate the dynamic threshold
    mean_score = np.mean(scores)
    std_dev = np.std(scores)
    dynamic_threshold = mean_score + std_multiplier * std_dev

    # Select nodes above the threshold
    selected_nodes = [
        node_id for node_id, score in node_scores.items() if score >= dynamic_threshold
    ]

    # If too many nodes were selected, keep only the best ones
    if len(selected_nodes) > max_nodes:
        selected_nodes = sorted(
            selected_nodes, key=lambda node_id: node_scores[node_id], reverse=True
        )[:max_nodes]

    # If not enough nodes were selected, add the next best ones to meet the quota
    if len(selected_nodes) < max_nodes:
        remaining_nodes = [
            node_id for node_id in node_scores if node_id not in selected_nodes
        ]
        remaining_nodes = sorted(
            remaining_nodes, key=lambda node_id: node_scores[node_id], reverse=True
        )
        selected_nodes += remaining_nodes[: max_nodes - len(selected_nodes)]

    return selected_nodes

## Step 5: Model Trainer 

Here we go. This is the big one. We'll now write the trainValModelCSVIMG function. This function is the conductor of our orchestra—it brings together the data, the model, the server, and the clients to run the entire federated learning simulation from start to finish.

Because it's so long and important, we'll build it in three parts.

### Part 1: Initialization and Starting the Training Loop

First, we'll define the function and set everything up. This includes:

Creating the global model, the Server, and all the Client objects.

Initializing a series of dictionaries to log every possible metric (loss, accuracy, selection scores, etc.) for every client and every round. This is crucial for analyzing the experiment later.

Starting the main training loop, which iterates through the communication rounds.

Inside the loop, we'll begin the first phase of a round: every client trains locally on the current global model.

### Part 2: Client Selection, Aggregation, and Global Evaluation
In this part of the trainValModelCSVIMG function, the server performs the following steps:

Evaluate: It uses all the metrics gathered from the clients to calculate the advanced performance scores (RF_loss, P_bias, etc.).

Select: Based on the chosen selection method (svmethod), it picks the top-performing clients for this round.

Aggregate: It averages the model updates (diffs) from only the selected clients to create a new, improved global model.

Evaluate Globally: It tests the new global model's performance on the entire held-out test set.

Save Best Model: If the new global model is the best one seen so far, its state is saved to a file.

### Part 3: Final Test and Saving Results
Now that the training is finished, we need to do two last things:

Load the best model that we saved during training and run a final, definitive test on it. This gives us the final performance numbers for our experiment.

Save all the logs we've been collecting into a CSV file. This is essential for creating plots and analyzing the training process, client behavior, and the effectiveness of the selection strategy.

# --- Phase 1: All clients perform local training ---
        for client_index in range(total_client):
            # Check if the client's data shape matches the server's global model input shape
            client_feature_count = clients[client_index].train_dataset.features.shape[1]
            global_model_input_features = server.global_model.csv_fc_1.in_features

            if client_feature_count != global_model_input_features:
                print(f"Skipping client {client_index} due to feature mismatch ({client_feature_count} features vs global model {global_model_input_features})")
                # Assign default/dummy values for this client's metrics for this round
                perEpoch_clients_losses[client_index] = [0]
                perEpoch_clients_train_acc[client_index] = 0
                perEpoch_clients_local_test_acc[client_index] = 0
                diff_client[client_index] = {name: torch.zeros_like(params) for name, params in server.global_model.state_dict().items()}
                perEpoch_clients_global_test_acc[client_index] = 0
                clients_train_acc[client_index].append(0)
                clients_train_loss[client_index].append(0)
                clients_test_acc[client_index].append(0)
                clients_epoch_selected[client_index].append(0)
                continue # Move to the next client

            # If shapes match, proceed with training
            round_client_model, diff, test_acc_client, loss_client, min_loss, max_loss, losses, train_acc = clients[client_index].local_train(server.global_model)
            
            # Store results for this client
            perEpoch_clients_losses[client_index] = losses
            perEpoch_clients_train_acc[client_index] = train_acc
            perEpoch_clients_local_test_acc[client_index] = test_acc_client
            diff_client[client_index] = diff
            
            # Evaluate this client's trained model on the SERVER's test set
            oneclient_global_test_acc, _ = validate(round_client_model, server.eval_loader, nn.CrossEntropyLoss())
            perEpoch_clients_global_test_acc[client_index] = oneclient_global_test_acc
            
            # Log the local and global test accuracies for this round
            clients_train_acc[client_index].append(test_acc_client)
            clients_train_loss[client_index].append(loss_client)
            clients_test_acc[client_index].append(oneclient_global_test_acc)
            clients_epoch_selected[client_index].append(0) # Mark as not selected (yet)

In [35]:
def trainValModelCSVIMG(model_name, svmethod, total_client, num_clients_to_select, epoch, max_acc, epoch_size, local_epoch_per_round, round_early_stop,
                        X_train_csv, X_train_img1, X_train_img2, Y_train,
                        X_test_csv, X_test_img1, X_test_img2, Y_test,
                        global_test_set, client_info, num_csv_features, num_classes,
                        client_modality_configs): # <-- NEW PARAM
    
    # --- 1. Initialization ---
    print(f"Initializing model and server for {total_client} multi-modal clients...")
    model_MLP = ModelCSVIMG(num_csv_features=num_csv_features, num_classes=num_classes)
    model_MLP = model_MLP.to(device)

    server = Server(model_MLP, epoch_size, global_test_set, num_clients_to_select)
    
    clients = []
    for i in range(total_client):
        client_model = ModelCSVIMG(num_csv_features=num_csv_features, num_classes=num_classes).to(device)
        train_dataset = [X_train_csv[i], X_train_img1[i], X_train_img2[i], Y_train[i]]
        val_dataset = [X_test_csv[i], X_test_img1[i], X_test_img2[i], Y_test[i]]
        
        clients.append(Client(model=client_model, 
                               epoch_size=epoch_size, 
                               local_epoch_per_round=local_epoch_per_round,
                               train_dataset=train_dataset,
                               val_dataset=val_dataset,
                               modality_config=client_modality_configs[i], # <-- Pass config
                               id=i))

    # --- Dictionaries for Logging ---
    # (Logging dictionaries setup is unchanged)
    clients_scoresDict = {}
    perEpoch_clients_losses = {}
    perEpoch_clients_train_acc = {}
    perEpoch_clients_local_test_acc = {}
    perEpoch_clients_global_test_acc = {}
    clients_train_acc = {}
    clients_train_loss = {}
    clients_test_acc = {}
    clients_test_loss = {}
    clients_rf_relative_loss_reduction = {}
    clients_rf_acc_train = {}
    clients_rf_acc_val = {}
    clients_rf_global_validation_accuracy = {}
    clients_rf_loss_outliers = {}
    clients_rf_performance_bias = {}
    clients_epoch_selected = {}

    for i in range(total_client + 1):
        clients_train_acc[i], clients_train_loss[i], clients_test_acc[i], clients_test_loss[i] = [], [], [], []
        clients_scoresDict[i], clients_rf_relative_loss_reduction[i], clients_rf_acc_train[i] = [], [], []
        clients_rf_acc_val[i], clients_rf_global_validation_accuracy[i], clients_rf_loss_outliers[i] = [], [], []
        clients_rf_performance_bias[i], clients_epoch_selected[i] = [], []

    
    epoch_count = 0
    # --- 2. Main Federated Learning Loop ---
    for e in range(epoch):
        print(f"--- Round {e+1}/{epoch} ---")
        if epoch_count >= round_early_stop:
            print("Early stopping triggered.")
            break
            
        diff_client = {}
        # (This accumulator is now created inside the server)

        # --- Phase 1: All clients perform local training ---
        for client_index in range(total_client):
            round_client_model, diff, test_acc_client, loss_client, min_loss, max_loss, losses, train_acc = clients[client_index].local_train(server.global_model)
            
            perEpoch_clients_losses[client_index] = losses
            perEpoch_clients_train_acc[client_index] = train_acc
            perEpoch_clients_local_test_acc[client_index] = test_acc_client
            diff_client[client_index] = diff
            
            # Evaluate this client's trained model on the GLOBAL test set
            # This eval uses ALL modalities, regardless of what the client trained on
            oneclient_global_test_acc, _ = validate(round_client_model, server.eval_loader, nn.CrossEntropyLoss(), {'csv': True, 'img1': True, 'img2': True})
            perEpoch_clients_global_test_acc[client_index] = oneclient_global_test_acc
            
            clients_train_acc[client_index].append(test_acc_client)
            clients_train_loss[client_index].append(loss_client)
            clients_test_acc[client_index].append(oneclient_global_test_acc)
            clients_epoch_selected[client_index].append(0)

        # --- Phase 2: Server evaluates, selects, and aggregates ---
        rf_relative_loss_reduction = calculate_relative_loss_reduction_as_list(perEpoch_clients_losses)
        rf_acc_train = calculate_relative_train_accuracy(perEpoch_clients_train_acc)
        rf_acc_val = calculate_relative_validation_accuracy(perEpoch_clients_local_test_acc)
        rf_global_validation_accuracy = calculate_global_validation_accuracy(perEpoch_clients_train_acc, perEpoch_clients_global_test_acc)
        rf_loss_outliers = calculate_loss_outliers(perEpoch_clients_losses)
        rf_performance_bias = calculate_performance_bias(perEpoch_clients_local_test_acc, perEpoch_clients_global_test_acc)
        
        # (Logging metrics is unchanged)
        for client_index in range(total_client):
            clients_rf_relative_loss_reduction[client_index].append(rf_relative_loss_reduction[client_index])
            clients_rf_acc_train[client_index].append(rf_acc_train[client_index])
            clients_rf_acc_val[client_index].append(rf_acc_val[client_index])
            clients_rf_global_validation_accuracy[client_index].append(rf_global_validation_accuracy[client_index])
            clients_rf_loss_outliers[client_index].append(rf_loss_outliers[client_index])
            clients_rf_performance_bias[client_index].append(rf_performance_bias[client_index])

        # --- Select clients ---
        candidates = []
        if svmethod == '5RF':
            candidates, scores = get_top_clients_with5RF(rf_relative_loss_reduction, rf_acc_train, rf_acc_val,
                                                         rf_global_validation_accuracy, rf_loss_outliers, rf_performance_bias,
                                                         num_clients_to_select)
        elif svmethod == 'pareto':
            candidates = pareto_optimization(rf_relative_loss_reduction, rf_acc_train, rf_acc_val,
                                              rf_global_validation_accuracy, rf_loss_outliers, rf_performance_bias,
                                              num_clients_to_select)
        elif svmethod == 'random':
            candidates = np.random.choice(total_client, num_clients_to_select, replace=False).tolist()

        print(f"Selected clients for aggregation: {candidates}")
        
        # Get the configs for *only* the selected clients
        selected_configs = {cid: client_modality_configs[cid] for cid in candidates}
        
        for selected_client_index in candidates:
            clients_epoch_selected[selected_client_index][-1] = 1
        
        # --- KEY STEP: Call new aggregation function ---
        server.model_aggregate(selected_configs, diff_client)

        # --- Phase 3: Evaluate the new global model ---
        acc, loss = server.model_eval()
        
        clients_test_acc[total_client].append(acc)
        clients_test_loss[total_client].append(loss)
        
        print(f"Round {e+1} Global Model - Accuracy: {acc:.2f}%, Loss: {loss:.4f}\\n")

        # (Saving logic is unchanged)
        epoch_count += 1
        if acc > max_acc:
            max_acc = acc
            print("New best model found! Saving model...")
            torch.save(server.global_model.state_dict(),
                       f"./acc_lossFiles/{model_name}_totalClient_{total_client}_NumClient_{num_clients_to_select}_epoch_{epoch}_svmethod_{svmethod}.pth")
            epoch_count = 0
        
    # --- Final evaluation ---
    print("\\n--- Final Evaluation on Best Model ---")
    model = ModelCSVIMG(num_csv_features=num_csv_features, num_classes=num_classes)
    model.load_state_dict(torch.load(
        f"./acc_lossFiles/{model_name}_totalClient_{total_client}_NumClient_{num_clients_to_select}_epoch_{epoch}_svmethod_{svmethod}.pth"))
    model = model.to(device)
    server.global_model = model # Load the best model into the server
    
    acc, loss = server.model_eval()

    print(f'Final Best Model Test Accuracy: {acc:.2f}%')
    print(f'Final Best Model Test Loss: {loss:.4f}')
    print(f'Max accuracy achieved during training: {max_acc:.2f}%')

    # --- Save all logged data to a CSV file ---
    # (CSV saving logic is unchanged)
    csv_file_name = f"./acc_lossFiles/{model_name}_totalClient_{total_client}_NumClient_{num_clients_to_select}_epoch_{epoch}_svmethod_{svmethod}.csv"
    header = ['client_id', 'client_name', 'Epoch', 'local_val_loss', 'local_val_accuracy', 'global_test_loss', 'global_test_accuracy',
              'rf_loss', 'rf_acc_train', 'rf_acc_val', 'rf_acc_global', 'p_loss', 'p_bias', 'selected']
    
    client_name_map = client_info
    client_name_map[total_client] = 'Global_Model'

    all_rows = []
    for i in range(total_client + 1):
        train_losses = clients_train_loss.get(i, [])
        train_accs = clients_train_acc.get(i, [])
        test_losses = clients_test_loss.get(i, [])
        test_accs = clients_test_acc.get(i, [])
        rf_losses = clients_rf_relative_loss_reduction.get(i, [])
        rf_acc_trains = clients_rf_acc_train.get(i, [])
        rf_acc_vals = clients_rf_acc_val.get(i, [])
        rf_acc_globals = clients_rf_global_validation_accuracy.get(i, [])
        p_losses = clients_rf_loss_outliers.get(i, [])
        p_biases = clients_rf_performance_bias.get(i, [])
        selecteds = clients_epoch_selected.get(i, [])

        max_epochs = max(len(lst) for lst in [
            train_losses, train_accs, test_losses, test_accs, rf_losses, 
            rf_acc_trains, rf_acc_vals, rf_acc_globals, p_losses, p_biases, selecteds
        ] if lst) # Added 'if lst' for safety

        for j in range(max_epochs):
            row_data = {
                'client_id': i,
                'client_name': client_name_map.get(i, ''),
                'Epoch': j + 1,
                'local_val_loss': train_losses[j] if j < len(train_losses) else '',
                'local_val_accuracy': train_accs[j] if j < len(train_accs) else '',
                'global_test_loss': test_losses[j] if j < len(test_losses) else '',
                'global_test_accuracy': test_accs[j] if j < len(test_accs) else '',
                'rf_loss': rf_losses[j] if j < len(rf_losses) else '',
                'rf_acc_train': rf_acc_trains[j] if j < len(rf_acc_trains) else '',
                'rf_acc_val': rf_acc_vals[j] if j < len(rf_acc_vals) else '',
                'rf_acc_global': rf_acc_globals[j] if j < len(rf_acc_globals) else '',
                'p_loss': p_losses[j] if j < len(p_losses) else '',
                'p_bias': p_biases[j] if j < len(p_biases) else '',
                'selected': selecteds[j] if j < len(selecteds) else ''
            }
            all_rows.append(row_data)

    results_df = pd.DataFrame(all_rows, columns=header)
    results_df.to_csv(csv_file_name, index=False)
    print(f"✅ Results successfully and safely saved to {csv_file_name}")

We've arrived at the final step! We have all the building blocks in place. The only thing left is to set our experimental parameters and create the main execution block that calls our functions and runs the simulation.

## Final Step: The main Function and Execution Block
This final piece of code does the following:

Sets Hyperparameters: Defines all the key variables for the experiment, like the number of clients, epochs, learning rate, etc. It also defines the different scenarios we want to test (e.g., different client selection methods, different data corruption scenarios).

Defines a main() function: This function orchestrates the experiment. It loads the client data, then loops through each experimental scenario. For scenarios involving "model loss," it intentionally corrupts the data for some clients (e.g., replacing their sensor data with random noise) to simulate system failures or unreliable participants.

Calls trainValModelCSVIMG: For each scenario, it calls our main training function to run a full federated learning simulation.

Executes main(): The standard if __name__ == "__main__": line ensures that the main function is called when you run the script.

In [34]:
# --- Define Experimental Scenarios and Hyperparameters ---
model_name = 'Heterogeneous_13_Clients' # NEW model name
svmethods = {'pareto', '5RF', 'random'}
svmethods = {'pareto'}  # For testing

# --- Hyperparameters ---
max_acc = 1
epoch = 50
epoch_size = 64
num_clients_to_select = 5 # Select 5 clients per round from the 13
local_epoch_per_round = 3
round_early_stop = 10
num_classes = 12 # 0-11
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Define paths
csv_path = '/home/syed/PhD/UP_Fall_Dataset/Sensor + Image/sensor.csv'
img_path = '/home/syed/PhD/UP_Fall_Dataset/Sensor + Image'

def main():
    # Load data using the new heterogeneous function
    # It now returns the configs list automatically
    X_train_csv, X_train_img1, X_train_img2, Y_train, \
    X_test_csv, X_test_img1, X_test_img2, Y_test, \
    global_test_set, client_info, num_csv_features, \
    client_modality_configs = load_heterogeneous_clients(
        csv_path=csv_path,
        img_path=img_path,
        num_classes=num_classes
    )
    
    total_client = len(client_info) # Get the actual number of clients created
    print(f"\\nTotal clients created: {total_client}")

    # Loop through each client selection method
    for svmethod in svmethods:
        print(f"\\n===== STARTING NEW EXPERIMENT: Model={model_name}, Selection={svmethod} =====")
        trainValModelCSVIMG(
            model_name, svmethod, total_client, num_clients_to_select, 
            epoch, max_acc, epoch_size, local_epoch_per_round, round_early_stop,
            X_train_csv, X_train_img1, X_train_img2, Y_train,
            X_test_csv, X_test_img1, X_test_img2, Y_test,
            global_test_set, client_info, num_csv_features, num_classes,
            client_modality_configs # Pass the configs from the data loader
        )

# This makes the script runnable
if __name__ == "__main__":
    if not os.path.exists("./acc_lossFiles"):
        os.makedirs("./acc_lossFiles")
    main()

--- Starting Heterogeneous Multi-Modal Data Loading ---
Loading all data sources...
All data sources loaded.
Found 258113 common timestamps across all 3 modalities.
Using 36 sensor features.
Training Clients (Subjects): [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(6), np.int64(7), np.int64(8), np.int64(10), np.int64(11), np.int64(12), np.int64(13)]
Test Set (Subjects): [np.int64(14), np.int64(15), np.int64(16), np.int64(17)]
Scaling and preprocessing data...
Creating client splits and assigning heterogeneous modalities...


/tmp/ipykernel_226133/149740253.py:160: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  Y_train_splits[client_id_counter] = Y_train[indices_train]


  - Client 0 (Subject 1): Modalities={'csv': True, 'img1': True, 'img2': True}
  - Client 1 (Subject 2): Modalities={'csv': True, 'img1': False, 'img2': False}
  - Client 2 (Subject 3): Modalities={'csv': False, 'img1': True, 'img2': True}
  - Client 3 (Subject 4): Modalities={'csv': True, 'img1': True, 'img2': False}
  - Client 4 (Subject 6): Modalities={'csv': True, 'img1': False, 'img2': True}
  - Client 5 (Subject 7): Modalities={'csv': False, 'img1': True, 'img2': False}
  - Client 6 (Subject 8): Modalities={'csv': False, 'img1': False, 'img2': True}
  - Client 7 (Subject 10): Modalities={'csv': True, 'img1': True, 'img2': True}
  - Client 8 (Subject 11): Modalities={'csv': True, 'img1': False, 'img2': False}
  - Client 9 (Subject 12): Modalities={'csv': False, 'img1': True, 'img2': True}
  - Client 10 (Subject 13): Modalities={'csv': True, 'img1': True, 'img2': False}
--- Successfully created 11 heterogeneous clients. ---
\nTotal clients created: 11
\n===== STARTING NEW EXPERIMEN

NameError: name 'trainValModelCSVIMG' is not defined